# Words in the model: one token vs cross-attention

Shapes only, random weights.


In [16]:
import sys

import torch
from einops import rearrange

from dream_robot.policies.verne.model import P, Verne

sys.path.insert(0, '..')  # nbs/probe.py
from probe import probe, show

torch.manual_seed(0)
net = Verne()
frame, state = torch.randn(1, 4, 16, 32), torch.randn(1, 8)  # canvas latent, joints + gripper
ins = torch.tensor([[0.0, 1, 0, 0, 1, 0]])  # WORDS one-hot: 'green', 'left'

## Now: the instruction is one of the video tokens


In [17]:
patches = rearrange(frame, 'b c (h p) (w q) -> b (h w) (p q c)', p=P, q=P)
parts = net.state_in(state)[:, None], net.instruction_in(ins)[:, None], net.patch_in(patches)
x = torch.cat(parts, 1) + net.video_pos
[p.shape for p in parts], x.shape

([torch.Size([1, 1, 256]), torch.Size([1, 1, 256]), torch.Size([1, 128, 256])],
 torch.Size([1, 130, 256]))

**130 video tokens of width 256: [state, instruction, 128 patches].**

- The whole instruction ('green' + 'left') is squeezed into **one** token, the same size as one patch.
- From here every layer is plain self-attention over these 130: a patch reaches the words only if it
  happens to give attention to token 1, competing with 129 other tokens.


## Now: how much of a patch's attention lands on the instruction (layer 0)


In [18]:
q, k, v = net.video[0].qkv_of(x)  # each [1, heads, 130 tokens, 32]
w = (q @ k.transpose(-1, -2) / 32**0.5).softmax(-1)  # [1, heads, 130 queries, 130 keys]
q.shape, w.shape, w[0, :, 2:, 1].mean().item(), 1 / 130

(torch.Size([1, 4, 130, 32]),
 torch.Size([1, 4, 130, 130]),
 0.0077002947218716145,
 0.007692307692307693)

**Each patch gives the instruction token 0.0077 of its attention, i.e. 1/130.**

- `w` is [1, 4 heads, 130 queries, 130 keys]: each row is one token's softmax over all 130 keys, summing to 1.
- At random init attention is near uniform, so the words get a 1-in-130 share. To use them, training
  must teach every patch's queries to single out key 1, against 128 patches and the state.


## Cross-attention: the words as their own rows


In [19]:
from torch import nn

word_in = nn.Embedding(6, 256)  # one learned row per word in WORDS
ids = ins.nonzero()[:, 1][None]  # [1, 2]: the words this prompt has
rows = word_in(ids)
ids, rows.shape

(tensor([[1, 4]]), torch.Size([1, 2, 256]))

**Word ids [1, 4] = 'green', 'left' → rows [1, 2, 256].**

- Two separate rows, not one summed token: 'green' and 'left' each keep their own vector.
- A T1 prompt has no words → 0 rows (to handle when packing: Verne attends to zero rows instead).


In [20]:
@probe
def _():
    show(
        ins,
        ins.shape,
        ins.nonzero(),
        ins.nonzero()[:, 1],
        ids,
        ids.shape,
        rows.shape,
    )

ins = tensor([[0., 1., 0., 0., 1., 0.]])
ins.shape = torch.Size([1, 6])
ins.nonzero() =
tensor([[0, 1],
        [0, 4]])
ins.nonzero()[:, 1] = tensor([1, 4])
ids = tensor([[1, 4]])
ids.shape = torch.Size([1, 2])
rows.shape = torch.Size([1, 2, 256])


## Cross-attention: patches ask, words answer


In [21]:
to_q, to_kv = nn.Linear(256, 4 * 32), nn.Linear(256, 2 * 4 * 32)
cq = rearrange(to_q(x[:, 2:]), 'b s (h d) -> b h s d', h=4)  # queries from the 128 patches
ck, cv = rearrange(to_kv(rows), 'b s (n h d) -> n b h s d', n=2, h=4)  # keys, values from words
cw = (cq @ ck.transpose(-1, -2) / 32**0.5).softmax(-1)  # [1, heads, 128 patches, 2 words]
cq.shape, ck.shape, cw.shape, cw[0, 0, 0]

(torch.Size([1, 4, 128, 32]),
 torch.Size([1, 4, 2, 32]),
 torch.Size([1, 4, 128, 2]),
 tensor([0.6119, 0.3881], grad_fn=<SelectBackward0>))

**Weights [1, 4 heads, 128 patches, 2 words]; patch 0, head 0: green 0.61, left 0.39.**

- Each patch's softmax is over the 2 words only, so all of it lands on the words (vs 1/130 in self-attention).
- Training no longer has to find the words; it only learns what to take from them (e.g. "green" → look for green).
- This is an extra sub-layer in each block: self-attention, then cross-attention to the words, then the MLP
  (Wan's block). The keys and values come from the same 2 rows in every layer.


In [22]:
@probe
def _():
    show(
        x.shape,
        x[:, 2:].shape,
        to_q(x[:, 2:]).shape,
        cq.shape,
        rows.shape,
        to_kv(rows).shape,
        ck.shape,
        cv.shape,
        cw.shape,
    )

x.shape = torch.Size([1, 130, 256])
x[:, 2:].shape = torch.Size([1, 128, 256])
to_q(x[:, 2:]).shape = torch.Size([1, 128, 128])
cq.shape = torch.Size([1, 4, 128, 32])
rows.shape = torch.Size([1, 2, 256])
to_kv(rows).shape = torch.Size([1, 2, 256])
ck.shape = torch.Size([1, 4, 2, 32])
cv.shape = torch.Size([1, 4, 2, 32])
cw.shape = torch.Size([1, 4, 128, 2])


## One block with cross-attention: 1) self-attention, without the instruction token


In [23]:
import torch.nn.functional as F

xs = torch.cat([x[:, :1], x[:, 2:]], 1)  # [state, 128 patches]: the words leave the sequence
b = net.video[0]
q, k, v = b.qkv_of(xs)
a = rearrange(F.scaled_dot_product_attention(q, k, v), 'b h s d -> b s (h d)')

xs = xs + b.out(a)
q.shape, a.shape, xs.shape

(torch.Size([1, 4, 129, 32]),
 torch.Size([1, 129, 128]),
 torch.Size([1, 129, 256]))

**129 tokens now: [state, 128 patches]; self-attention keeps the shape [1, 129, 256].**

- The same block as today (`net.video[0]`: its norm, q k v, output layer), just one token fewer.


## 2) cross-attention to the words, then the MLP


In [24]:
norm3, cross_out = nn.LayerNorm(256), nn.Linear(4 * 32, 256)
cq = rearrange(to_q(norm3(xs)), 'b s (h d) -> b h s d', h=4)  # all 129 tokens ask
a = F.scaled_dot_product_attention(cq, ck, cv)  # each token's weighted mix of the 2 word values
xs = xs + cross_out(rearrange(a, 'b h s d -> b s (h d)'))
xs = xs + b.mlp(b.norm2(xs))
cq.shape, ck.shape, a.shape, xs.shape

(torch.Size([1, 4, 129, 32]),
 torch.Size([1, 4, 2, 32]),
 torch.Size([1, 4, 129, 32]),
 torch.Size([1, 129, 256]))

**Cross-attention: queries [1, 4, 129, 32] × keys [1, 4, 2, 32] → each token's mix a [1, 4, 129, 32]; out [1, 129, 256].**

- Same shape out as in: cross-attention adds to each token (residual), it doesn't add tokens.
- New pieces per block: `norm3`, `to_q`, `to_kv`, `cross_out` (Wan's block: self-attention → cross-attention → MLP).
- The video sequence is never longer than today; the words cost 2 extra keys per token per layer.


## Action side now: the words as one vector added to every action token


In [25]:
chunk, tau = torch.randn(1, 16, 8), torch.rand(1)  # a noisy chunk of 16 actions, its flow time
y = net.action_in(chunk) + net.action_pos + net.time_in(tau[:, None])[:, None]
w_act = net.instruction_act(ins)[:, None]  # [1, 1, 128], broadcast over the 16 steps
y.shape, w_act.shape, (y + w_act).shape

(torch.Size([1, 16, 128]), torch.Size([1, 1, 128]), torch.Size([1, 16, 128]))

**Action tokens [1, 16, 128]; the words [1, 1, 128], the same vector added to all 16 steps.**

- Step 1 and step 16 of the chunk get identical word information, fixed before any layer runs.


## Action side with cross-attention: each action step asks the words


In [26]:
act_rows = nn.Linear(
    256, 128
)  # the action side's own view of the same word rows (Verne: one per side)
a_q, a_kv = nn.Linear(128, 4 * 32), nn.Linear(128, 2 * 4 * 32)
aq = rearrange(a_q(y), 'b s (h d) -> b h s d', h=4)  # queries from the 16 action tokens
ak, av = rearrange(a_kv(act_rows(rows)), 'b s (n h d) -> n b h s d', n=2, h=4)
aw = (aq @ ak.transpose(-1, -2) / 32**0.5).softmax(-1)  # [1, heads, 16 steps, 2 words]
aq.shape, ak.shape, aw.shape, aw[0, 0, :3]

(torch.Size([1, 4, 16, 32]),
 torch.Size([1, 4, 2, 32]),
 torch.Size([1, 4, 16, 2]),
 tensor([[0.5000, 0.5000],
         [0.5440, 0.4560],
         [0.4644, 0.5356]], grad_fn=<SliceBackward0>))

**Weights [1, 4 heads, 16 steps, 2 words]; head 0, steps 0-2: 0.50/0.50, 0.54/0.46, 0.46/0.54.**

- Each action step weighs the words on its own, and again in every layer (its queries change as it denoises).
- The words reach the actions directly, not only through the video side's keys and values.
- New pieces per action block: `act_rows` (once, shared by the blocks), a norm, `a_q`, `a_kv`, an output layer.


## Packing: word rows for a batch, when a prompt can have no words


In [27]:
batch = torch.tensor([[0.0, 1, 0, 0, 1, 0], [0.0, 0, 0, 0, 0, 0]])  # T5 'green left', T1 none
rows_b = batch[:, :, None] * word_in.weight  # [2, 6, 256]: every word a row, absent ones zero
show(rows_b.shape, rows_b.norm(dim=-1))

rows_b.shape = torch.Size([2, 6, 256])
rows_b.norm(dim=-1) =
tensor([[ 0.0000, 15.0306,  0.0000,  0.0000, 16.4069,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],
       grad_fn=<LinalgVectorNormBackward0>)


**rows [2 prompts, 6 words, 256]: one row per word in WORDS, zero where the prompt lacks it.**

- T5 'green left': rows 1 and 4 are real (norm ~16), the other 4 zero. T1: all 6 zero.
- Every prompt has the same number of rows, so the batch stacks with no mask (Verne pads with zero rows too).


## What the patches read from them


In [ ]:
bk, bv = rearrange(to_kv(rows_b), 'b s (n h d) -> n b h s d', n=2, h=4)  # [2, 4, 6, 32]
bq = cq[:, :, :1].expand(2, -1, -1, -1)  # the same query (token 0) for both prompts
bw = (bq @ bk.transpose(-1, -2) / 32**0.5).softmax(-1)  # [2, heads, 1, 6 words]
show(bw[:, 0, 0])

bw[:, 0, 0] =
tensor([[0.1929, 0.0545, 0.1929, 0.1929, 0.1739, 0.1929],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667]],
       grad_fn=<SelectBackward0>)


**T5 prompt: the 4 zero rows take 0.19 each = 0.77 of the attention; green 0.05, left 0.17. T1: 1/6 each.**

- Works: T1 gets a defined answer (a zero row's key and value are just the layer's bias), no NaN.
- Costs: the zero rows are 4 identical keys, so together they soak up most of a T5 patch's attention
  at init. Training can learn to push them down, but it starts diluted.
- The other way: keep only real words (masked) plus one learned "no word" row, so every prompt has ≥ 1 key.


In [ ]:
@probe
def _():
    show(
        rows_b.shape,
        to_kv(rows_b).shape,
        256 / 8,
        cq.shape,
        cq[:, :, :1].shape,
        cq[:, :, :1].expand(2, -1, -1, -1).shape,
        bq.shape,
        bk.shape,
        bw.shape,
    )

rows_b.shape = torch.Size([2, 6, 256])
to_kv(rows_b).shape = torch.Size([2, 6, 256])
256 / 8 = 32.0
cq.shape = torch.Size([1, 4, 129, 32])
cq[:, :, :1].shape = torch.Size([1, 4, 1, 32])
cq[:, :, :1].expand(2, -1, -1, -1).shape = torch.Size([2, 4, 1, 32])
bq.shape = torch.Size([2, 4, 1, 32])
bk.shape = torch.Size([2, 4, 6, 32])
bw.shape = torch.Size([2, 4, 1, 6])


## The fix: only real words (masked) + one learned "no word" row every prompt has

In [34]:
no_word = nn.Parameter(torch.randn(1, 1, 256))  # key 0 for every prompt, so T1 has one too
rows_m = torch.cat([no_word.expand(2, -1, -1), batch[:, :, None] * word_in.weight], 1)
mask = torch.cat([torch.ones(2, 1, dtype=torch.bool), batch.bool()], 1)  # True = may attend
show(rows_m.shape, mask)

rows_m.shape = torch.Size([2, 7, 256])
mask =
tensor([[ True, False,  True, False, False,  True, False],
        [ True, False, False, False, False, False, False]])


**rows [2, 7, 256] = [no word, 6 words]; mask True at key 0 always, then at the prompt's words.**
- T5 'green left': keys 0, 2 (green), 5 (left). T1: key 0 only.
- Absent words are still rows (zeros) so the batch stacks, but the mask hides them.

## What the patches read now

In [35]:
mk, mv = rearrange(to_kv(rows_m), 'b s (n h d) -> n b h s d', n=2, h=4)  # [2, 4, 7, 32]
scores = bq @ mk.transpose(-1, -2) / 32**0.5  # [2, heads, 1, 7]
mw = scores.masked_fill(~mask[:, None, None], float('-inf')).softmax(-1)
show(mw[:, 0, 0])

mw[:, 0, 0] =
tensor([[0.4593, 0.0000, 0.1290, 0.0000, 0.0000, 0.4116, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<SelectBackward0>)


**T5: no word 0.46, green 0.13, left 0.41, absent words 0. T1: no word 1.0.**
- No dilution from absent words: a T5 patch splits its attention over 3 keys, not 6 identical zero rows.
- T1 reads only the learned "no word" row: a fixed, learnable "no instruction" signal, no NaN.
- "No word" is also a place for a T5 patch to park attention when the words don't matter to it.

## Pack 1: a block with the cross-attention sub-layer (Wan's order: self → cross → MLP)

In [36]:
from dream_robot.policies.verne.model import HEAD_DIM, HEADS, Block


class CrossBlock(Block):
    def __init__(self, dim: int):
        super().__init__(dim)
        self.norm3, self.cq = nn.LayerNorm(dim), nn.Linear(dim, HEADS * HEAD_DIM)
        self.ckv, self.cout = nn.Linear(dim, 2 * HEADS * HEAD_DIM), nn.Linear(HEADS * HEAD_DIM, dim)

    def finish(self, x, q, k, v, rows, rmask, mask=None):
        # rows [B, R, dim] the word rows, rmask [B, R] True = may attend; mask as Block's
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        x = x + self.out(rearrange(a, 'b h s d -> b s (h d)'))
        cq = rearrange(self.cq(self.norm3(x)), 'b s (h d) -> b h s d', h=HEADS)
        ck, cv = rearrange(self.ckv(rows), 'b s (n h d) -> n b h s d', n=2, h=HEADS)
        a = F.scaled_dot_product_attention(cq, ck, cv, attn_mask=rmask[:, None, None])
        x = x + self.cout(rearrange(a, 'b h s d -> b s (h d)'))
        return x + self.mlp(self.norm2(x))

## One CrossBlock on the batch of 2 prompts

In [37]:
cb = CrossBlock(256)
x2 = xs.detach().expand(2, -1, -1)  # [2, 129, 256]: the same tokens under both prompts
q, k, v = cb.qkv_of(x2)
out = cb.finish(x2, q, k, v, rows_m, mask)
n_new = sum(p.numel() for p in cb.parameters()) - sum(p.numel() for p in net.video[0].parameters())
show(out.shape, n_new, (out[0] - out[1]).abs().max())

out.shape = torch.Size([2, 129, 256])
n_new = 132224
(out[0] - out[1]).abs().max() = tensor(1.1358, grad_fn=<MaxBackward1>)


**Out [2, 129, 256], same as in; the two prompts now give different tokens (max diff 1.14); +132k params per video block.**
- Same patches, different words → different outputs: the words reach every token inside the block.
- Extra per video block: norm3 + `cq` (256→128) + `ckv` (256→256) + `cout` (128→256) = 132,224.
  Action blocks (width 128) add 66k each: about 0.8M over the 8 blocks, on top of the current model.

## Pack 2: the model with words as cross-attention rows on both sides (no future branch)

In [38]:
from dream_robot.policies.verne.model import CHUNK, D_ACTION, D_VIDEO, LAYERS, N_PATCH, Z


class VerneX(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_in, self.state_in = nn.Linear(Z * P * P, D_VIDEO), nn.Linear(8, D_VIDEO)
        self.video_pos = nn.Parameter(
            torch.randn(1, 1 + N_PATCH, D_VIDEO) * 0.02
        )  # [state, patches]
        self.video_words = nn.Embedding(7, D_VIDEO)  # row 0 'no word', then WORDS
        self.video = nn.ModuleList([CrossBlock(D_VIDEO) for _ in range(LAYERS)])
        self.action_in, self.action_words = nn.Linear(8, D_ACTION), nn.Embedding(7, D_ACTION)
        self.time_in = nn.Sequential(
            nn.Linear(1, D_ACTION), nn.SiLU(), nn.Linear(D_ACTION, D_ACTION)
        )
        self.action_pos = nn.Parameter(torch.randn(1, CHUNK, D_ACTION) * 0.02)
        self.action = nn.ModuleList([CrossBlock(D_ACTION) for _ in range(LAYERS)])
        self.action_out = nn.Linear(D_ACTION, 8)

    def see(self, frame, state, ins):
        # -> each video layer's keys and values; the words enter only through cross-attention
        patches = rearrange(frame, 'b c (h p) (w q) -> b (h w) (p q c)', p=P, q=P)
        x = torch.cat([self.state_in(state)[:, None], self.patch_in(patches)], 1) + self.video_pos
        rows = self.video_words.weight.expand(len(x), -1, -1)  # [B, 7, D_VIDEO], all 7 rows
        rmask = torch.cat([torch.ones_like(ins[:, :1]), ins], 1).bool()  # 'no word' + the prompt's
        kvs = []
        for block in self.video:
            q, k, v = block.qkv_of(x)
            kvs.append((k, v))
            x = block.finish(x, q, k, v, rows, rmask)
        return kvs

    def velocity(self, kvs, x, tau, ins):
        y = self.action_in(x) + self.action_pos + self.time_in(tau[:, None])[:, None]
        rows = self.action_words.weight.expand(len(y), -1, -1)  # [B, 7, D_ACTION]
        rmask = torch.cat([torch.ones_like(ins[:, :1]), ins], 1).bool()
        for block, (k_video, v_video) in zip(self.action, kvs):
            q, k, v = block.qkv_of(y)
            k, v = torch.cat([k_video, k], 2), torch.cat([v_video, v], 2)
            y = block.finish(y, q, k, v, rows, rmask)
        return self.action_out(y)

## VerneX vs today's model: shapes and size

In [39]:
nx = VerneX()
kvs = nx.see(frame.expand(2, -1, -1, -1), state.expand(2, -1), batch)
vel = nx.velocity(kvs, torch.randn(2, CHUNK, 8), torch.rand(2), batch)
today = sum(p.numel() for name, p in net.named_parameters() if not name.startswith('future'))
show(len(kvs), kvs[0][0].shape, vel.shape, today, sum(p.numel() for p in nx.parameters()))

len(kvs) = 4
kvs[0][0].shape = torch.Size([2, 4, 129, 32])
vel.shape = torch.Size([2, 16, 8])
today = 3489928
sum(p.numel() for p in nx.parameters()) = 4283784


**4 video layers of keys/values [2, 4, 129, 32] (was 130 tokens); velocity [2, 16, 8] as today; 4.28M params vs 3.49M (+0.79M, +23%).**
- The action side's interface is unchanged: it still reads each video layer's keys and values, plus the words itself.
- Gone: `instruction_in` (the word token), `instruction_act` (the vector added to actions). New: two 7-row
  word tables and the cross-attention sub-layer in all 8 blocks.
- The +23% is almost all cross-attention projections; to compare fairly, the step 08 baseline is 22/50 at 3.49M.